In [42]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from tqdm import tqdm

In [41]:
! pip install tqdm

In [6]:
# exemplo de proxima página: https://fnpetroleiros.org.br/category/direitos/page/3/
# o padrão de ir para próxima página é URL + 'page/2/'

In [ ]:
def get_html(url):
    payload = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Referer": "https://google.com"
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content


In [20]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        span = article.find('span')
        date = span.text.strip()
        date = datetime.datetime.strptime(date, '%d/%m/%Y')
        link_date = [link, date]
        news_links.append(link_date)

    return news_links

In [32]:
def get_validated_links(news_links, min_date = datetime.datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page 

In [27]:
def get_next_page(fnp_url, next_page_number = 1):
    validated_news_links = []
    url = fnp_url + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [33]:
fnp_urls = [
    'https://fnpetroleiros.org.br/category/diversos/',
    'https://fnpetroleiros.org.br/category/post_category/',
    'https://fnpetroleiros.org.br/category/artigos/',
    'https://fnpetroleiros.org.br/category/petrobras/',
    'https://fnpetroleiros.org.br/category/brasil/',
    'https://fnpetroleiros.org.br/category/mundo/',
    'https://fnpetroleiros.org.br/category/eleicoes/',
    'https://fnpetroleiros.org.br/category/direitos/'
]

next_page_number = 1
validated_news_links = []
for fnp_url in fnp_urls:
    url = fnp_url + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

len(validated_news_links)

c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.htm

25

In [34]:
validated_news_links

[['https://fnpetroleiros.org.br/da-execucao-individual-de-acoes-coletivas-a-valorizacao-da-advocacia-sindical/',
  datetime.datetime(2025, 8, 5, 0, 0)],
 ['https://fnpetroleiros.org.br/nota-de-pesar-estudantes-da-ufpa-morrem-em-acidente-na-br-153/',
  datetime.datetime(2025, 7, 16, 0, 0)],
 ['https://fnpetroleiros.org.br/nota-de-pesar-eduardo-de-melo-brito-presente/',
  datetime.datetime(2025, 7, 7, 0, 0)],
 ['https://fnpetroleiros.org.br/representantes-da-fnp-e-da-fup-se-reunem-com-embaixador-do-ira-e-estabelecem-um-canal-de-dialogo-sobre-relacoes-trabalhistas-e-soberania-energetica/',
  datetime.datetime(2025, 7, 2, 0, 0)],
 ['https://fnpetroleiros.org.br/grave-petrobras-abandona-negociacao-do-plano-de-cargos-apos-censurar-gravacoes-de-videos-da-fnp/',
  datetime.datetime(2025, 6, 25, 0, 0)],
 ['https://fnpetroleiros.org.br/alteracao-na-clt-protege-familias-de-trabalhadores-transferidos-no-setor-publico/',
  datetime.datetime(2025, 8, 6, 0, 0)],
 ['https://fnpetroleiros.org.br/entida

In [ ]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find_all('p')

    return title, paragraphs

In [ ]:
result = []
for url, date in tqdm(validated_news_links):
    title, paragraphs = get_content_news(url)

    num_paragraph = 1
    for paragraph in paragraphs:
        result.append(
            {
                'sindicato': 'FNP',
                'url' : url,
                'titulo' : title,
                'data': str(validated_news_links[0][1]).split(' ')[0],
                'paragrafo' : paragraph.text,
                'num_paragrafo' : num_paragraph
            }
        )
        num_paragraph += 1

  0%|                                                                                                                                                                                     | 0/25 [00:00<?, ?it/s]c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
  0%|                                                                                                                                                                                     | 0/25 [00:01<?, ?it/s]


In [48]:
result

[{'sindicato': 'FNP',
  'url': 'https://fnpetroleiros.org.br/da-execucao-individual-de-acoes-coletivas-a-valorizacao-da-advocacia-sindical/',
  'titulo': 'Da Execução Individual de Ações Coletivas – A Valorização da Advocacia Sindical',
  'data': '2025-08-05',
  'paragrafo': 'Por Grupo Jurídico da FNP',
  'num_paragrafo': 1},
 {'sindicato': 'FNP',
  'url': 'https://fnpetroleiros.org.br/da-execucao-individual-de-acoes-coletivas-a-valorizacao-da-advocacia-sindical/',
  'titulo': 'Da Execução Individual de Ações Coletivas – A Valorização da Advocacia Sindical',
  'data': '2025-08-05',
  'paragrafo': '\xa0',
  'num_paragrafo': 2},
 {'sindicato': 'FNP',
  'url': 'https://fnpetroleiros.org.br/da-execucao-individual-de-acoes-coletivas-a-valorizacao-da-advocacia-sindical/',
  'titulo': 'Da Execução Individual de Ações Coletivas – A Valorização da Advocacia Sindical',
  'data': '2025-08-05',
  'paragrafo': 'A advocacia sindical exige do profissional não apenas profundo conhecimento das leis tra

In [ ]:
#formato json
[{
    'sindicato': 'str',
    'url' : 'str',
    'titulo' : 'str',
    'data':'AAAA-MM-DD',
    'paragrafo' : 'str1'
    'num_paragrafo' : 'int'
},
{
    'sindicato': 'str',
    'url' : 'str',
    'titulo' : 'str',
    'data':'AAAA-MM-DD',
    'paragrafo' : 'str2'
    'num_paragrafo' : 'int'
}]

def obtain_links(url, min_date = datetime.datetime(2025,6,1)):
    def get_next_url(current_url):
        if current_url[-2].isnumeric():
            next_page_num = int(current_url[-2])+1
            return 'https://fup.org.br/noticias/'+str(next_page_num)+'/'
        else:
            return 'https://fup.org.br/noticias/2/'
    
    dict_links = {}
    
    while True:
        print('lendo', url, 'para obter links')
        soup = BeautifulSoup(requests.get(url).text, 'html.parser')
        datas = [span.get_text().replace('\n','').replace('\t','') for span in soup.find_all('span', class_='elementor-icon-list-text elementor-post-info__item elementor-post-info__item--type-date')]
        datas_datetime = [datetime.datetime.strptime(date.split(',')[-1].strip(), '%d %B %Y %H:%M') for date in datas]
        links = [h1_a[0]['href'] for h1_a in [h1.find_all('a', href = True) for h1 in soup.find_all('h1', class_='elementor-heading-title elementor-size-default')] if len(h1_a)>0]
        dict_date_links = {data:link for data, link in zip(datas_datetime,links)}
        for data, link in dict_date_links.items():
            if data < min_date:
                return dict_links
            else:
                if link not in dict_links:
                    dict_links.update({link:{'data':data}})
        url = get_next_url(url)
    return dict_links

def get_links_text(min_paragraph_len = 14,dict_links = None):
    if dict_links == None:
        dict_links = obtain_links(url)
    print ('obtendo textos de', len(dict_links), 'links')
    for link, link_metadata in tqdm(links.items()):
        lst_paragraphs = []
        soup = BeautifulSoup(requests.get(link).text, 'html.parser')
        titulo = soup.find('h1', class_='elementor-heading-title elementor-size-default').get_text()
        resultsets_paragraphs = [div for div in [div.find_all('p', style=None) for div in soup.find_all('div', class_='elementor-widget-container')] if len(div) > 0]
        for resultset in resultsets_paragraphs:
            for paragraph in resultset:
                if len(paragraph.find_all('em')) > 0:
                    continue #evita que textos de comentários estejam entre os ingeridos
                text = paragraph.get_text()
                if len(text) >= min_paragraph_len:
                    lst_paragraphs.append(text)
        link_metadata.update({'titulo':titulo})
        link_metadata.update({'paragraphs':lst_paragraphs})
        dict_links[link] = link_metadata
    return dict_links



#links = obtain_links(url)
links_paragrafos = get_links_text(dict_links = links)

In [23]:
url = 'https://fnpetroleiros.org.br/category/diversos/page/99/'
payload = {}
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
    "Referer": "https://google.com"
}
response = requests.request("GET", url, headers=headers, data=payload, verify=False)
html_content = response.text

c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [24]:
html_content

'<!DOCTYPE html><html lang="pt-BR"><head><meta charset="UTF-8" /><meta name="viewport" content="width=device-width, initial-scale=1" /><meta name=\'robots\' content=\'noindex, follow\' /> <style>img:is([sizes="auto" i], [sizes^="auto," i]) { contain-intrinsic-size: 3000px 1500px }</style> <!-- This site is optimized with the Yoast SEO plugin v25.7 - https://yoast.com/wordpress/plugins/seo/ --><meta property="og:locale" content="pt_BR" /><meta property="og:title" content="Página não encontrada - Federação Nacional dos Petroleiros" /><meta property="og:site_name" content="Federação Nacional dos Petroleiros" /> <script type="application/ld+json" class="yoast-schema-graph">{"@context":"https://schema.org","@graph":[{"@type":"WebSite","@id":"https://fnpetroleiros.org.br/#website","url":"https://fnpetroleiros.org.br/","name":"Federação Nacional dos Petroleiros","description":"","publisher":{"@id":"https://fnpetroleiros.org.br/#organization"},"potentialAction":[{"@type":"SearchAction","target

In [ ]:
response

<Response [404]>

: 

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from tqdm import tqdm

def get_html(url):
    payload = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Referer": "https://google.com"
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        span = article.find('span')
        date = span.text.strip()
        date = datetime.datetime.strptime(date, '%d/%m/%Y')
        link_date = [link, date]
        news_links.append(link_date)

    return news_links

def get_validated_links(news_links, min_date = datetime.datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

def get_next_page(fnp_url, next_page_number = 1):
    validated_news_links = []
    url = fnp_url + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find_all('p')

    return title, paragraphs

def main():
    fnp_urls = [
        'https://fnpetroleiros.org.br/category/diversos/',
        'https://fnpetroleiros.org.br/category/post_category/',
        'https://fnpetroleiros.org.br/category/artigos/',
        'https://fnpetroleiros.org.br/category/petrobras/',
        'https://fnpetroleiros.org.br/category/brasil/',
        'https://fnpetroleiros.org.br/category/mundo/',
        'https://fnpetroleiros.org.br/category/eleicoes/',
        'https://fnpetroleiros.org.br/category/direitos/'
    ]

    # Getting News Links

    next_page_number = 1
    validated_news_links = []
    for fnp_url in fnp_urls:
        url = fnp_url + 'page/' + str(next_page_number) + '/'
        html_content = get_html(url)
        news_links = get_links_and_dates(html_content)
        validated_links, next_page = get_validated_links(news_links)

        for validated_link in validated_links:
            validated_news_links.append(validated_link)

        if next_page:
            validated_links = get_next_page(fnp_url, next_page_number + 1)
            
            for validated_link in validated_links:
                validated_news_links.append(validated_link)

    # Getting Paragraphs

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)

        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'FNP',
                    'url' : url,
                    'titulo' : title,
                    'data': str(validated_news_links[0][1]).split(' ')[0],
                    'paragrafo' : paragraph.text,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [19]:
from PetrobrasChat.Scrapers import scraper_fnp

import os
os.listdir()

['.venv', '.vscode', 'PetrobrasChat']

In [ ]:

result = main()

df = pd.DataFrame(result)

df

c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fnpetroleiros.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.htm

,sindicato,url,titulo,data,paragrafo,num_paragrafo
0,FNP,https://fnpetroleiros.org.br/da-execucao-indiv...,Da Execução Individual de Ações Coletivas – A ...,2025-08-05,Por Grupo Jurídico da FNP,1
1,FNP,https://fnpetroleiros.org.br/da-execucao-indiv...,Da Execução Individual de Ações Coletivas – A ...,2025-08-05,,2
2,FNP,https://fnpetroleiros.org.br/da-execucao-indiv...,Da Execução Individual de Ações Coletivas – A ...,2025-08-05,A advocacia sindical exige do profissional não...,3
3,FNP,https://fnpetroleiros.org.br/da-execucao-indiv...,Da Execução Individual de Ações Coletivas – A ...,2025-08-05,O(A) advogado(a) sindical atua tanto em demand...,4
4,FNP,https://fnpetroleiros.org.br/da-execucao-indiv...,Da Execução Individual de Ações Coletivas – A ...,2025-08-05,"É nessa etapa final que, infelizmente, surgem ...",5
...,...,...,...,...,...,...
511,FNP,https://fnpetroleiros.org.br/terceirizacao-com...,Terceirização como projeto de destruição: resi...,2025-08-05,"“É a empresa que terceiriza, que precariza, qu...",19
512,FNP,https://fnpetroleiros.org.br/terceirizacao-com...,Terceirização como projeto de destruição: resi...,2025-08-05,“Não há solução justa dentro do modelo atual. ...,20
513,FNP,https://fnpetroleiros.org.br/terceirizacao-com...,Terceirização como projeto de destruição: resi...,2025-08-05,"Paralelamente ao painel sobre terceirização, o...",21
514,FNP,https://fnpetroleiros.org.br/terceirizacao-com...,Terceirização como projeto de destruição: resi...,2025-08-05,Todos os direitos reservados @ 2022,22
